# Machine Doctor — Deep Learning Add-on
## Step 2: Preprocess into Labeled Training Windows

**Key design decision:** we split into train/test **by file**, not by individual window. Windows cut from the same recording are highly correlated (near-duplicates) -- if some ended up in training and others in testing, the network could get artificially high accuracy by recognizing the recording itself rather than the actual fault pattern. Splitting by file first guarantees the test set is genuinely unseen data.

In [ ]:
!pip install -q kagglehub
import kagglehub, os, glob
import numpy as np
import scipy.io

dataset_path = kagglehub.dataset_download("esraakhaled299/cwru-data")
print("Dataset at:", dataset_path)

In [ ]:
# Gather every .mat file and label it based on which folder it's in.
# We use the 12k Drive-End data specifically -- it's the most complete
# and widely-benchmarked subset (see the CWRU dataset docs).
CLASS_NAMES = ["Normal", "Ball", "InnerRace", "OuterRace"]

all_files = glob.glob(os.path.join(dataset_path, "**", "*.mat"), recursive=True)
de_files = [f for f in all_files if "12k_DE" in f or "12k" in f]
if not de_files:
    de_files = all_files  # fallback if folder naming differs slightly

labeled_files = []
for f in de_files:
    for class_name in CLASS_NAMES:
        if class_name in f:
            labeled_files.append((f, class_name))
            break

print(f"Labeled {len(labeled_files)} files")
for class_name in CLASS_NAMES:
    count = sum(1 for _, c in labeled_files if c == class_name)
    print(f"  {class_name}: {count} files")

In [ ]:
# Split by FILE first (the leakage-avoidance step described above),
# stratified so each class is represented proportionally in both splits.
import random
random.seed(42)

train_files, test_files = [], []
for class_name in CLASS_NAMES:
    class_files = [f for f, c in labeled_files if c == class_name]
    random.shuffle(class_files)
    split_idx = max(1, int(len(class_files) * 0.8))
    train_files += [(f, class_name) for f in class_files[:split_idx]]
    test_files += [(f, class_name) for f in class_files[split_idx:]]

print(f"Train files: {len(train_files)}, Test files: {len(test_files)}")
assert set(f for f, c in train_files).isdisjoint(f for f, c in test_files), "LEAKAGE: a file is in both splits!"
print("Confirmed: zero overlap between train and test files.")

In [ ]:
# Cut each file into fixed-length windows ("flashcards"). 1024 samples at
# 12kHz is about 85ms -- long enough to contain several fault impulses,
# short enough to give us many training examples per file.
WINDOW_SIZE = 1024
STRIDE = 512  # 50% overlap -- more training examples, safe since overlap
              # never crosses the train/test file boundary

def load_de_signal(filepath):
    mat = scipy.io.loadmat(filepath)
    key = [k for k in mat.keys() if "DE_time" in k][0]
    return mat[key].flatten()

def make_windows(file_label_list):
    X, y = [], []
    for filepath, class_name in file_label_list:
        signal = load_de_signal(filepath)
        for start in range(0, len(signal) - WINDOW_SIZE, STRIDE):
            window = signal[start:start + WINDOW_SIZE]
            # Per-window normalization (zero mean, unit variance) --
            # makes the network robust to overall vibration amplitude
            # differences between recordings, focusing it on SHAPE/pattern.
            window = (window - window.mean()) / (window.std() + 1e-8)
            X.append(window)
            y.append(CLASS_NAMES.index(class_name))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

X_train, y_train = make_windows(train_files)
X_test, y_test = make_windows(test_files)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print()
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {np.sum(y_train==i)} train windows, {np.sum(y_test==i)} test windows")

In [ ]:
# Visual sanity check: plot one example window per class
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, name in enumerate(CLASS_NAMES):
    idx = np.where(y_train == i)[0][0]
    axes[i].plot(X_train[idx], linewidth=0.7)
    axes[i].set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# Save the processed dataset to Google Drive so it survives between
# Colab sessions -- Step 3 (model training) will just load this file
# directly instead of reprocessing everything from scratch.
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/machine_doctor_dl'
os.makedirs(save_dir, exist_ok=True)

np.savez(
    os.path.join(save_dir, 'processed_data.npz'),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    class_names=CLASS_NAMES,
)
print(f"Saved to {save_dir}/processed_data.npz")